# Degree-2 Moon raypaths with `pykonal`

This notebook keeps the moonquake/station catalog logic from `RayPath.ipynb`, but computes travel times and raypaths with `pykonal` on a simple 3D spherical Moon.

What is reused from `RayPath.ipynb`:
- the Apollo station dictionary,
- the selected moonquake table format (`event`, `station`, `epi_deg`, `lat`, `Lon`),
- the `vpremoon` reader,
- the plotting idea: show raypaths and show travel times for the selected events.

Notes:
- The notebook first tries to load the existing `selected_events_...csv` that already contains source coordinates.
- If that file is missing, it falls back to the Excel selection logic from `RayPath.ipynb`.
- The notebook now calls the native `pykonal.trace_ray()` first. If the native tracer returns a degenerate path, the code can either fail loudly or fall back to the older gradient-based tracer for debugging.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pykonal
from IPython.display import display


RMOON_KM = 1737.1
SOURCE_DEPTH_KM = 30.0
PHASE_NAME = "P"

MODEL_FILE = Path("/Users/ramonmargarit/Desktop/vpremoon_mantle_closed.tvel")
SELECTED_EVENTS_CSV = Path(
    "/Users/ramonmargarit/Phd/Projects/Heterogeneities_Mantle/Heterogeneities-project/data/processed/selected_events/selected_events_best_7_bands_fixed_hold0_LOW0p75_MINPOST1_KNEG5_KPREPOS2.csv"
)
XLSX = Path("/Users/ramonmargarit/IPGP Dropbox/Ramon Margarit/PhD/Science/Modelling_Envelopes/data/processed/Shallow_processed_RESULTS.xlsx")
SHEET = "best_7_bands_fixed_hold0"

FC = 5.0
BANDS = np.array([3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0])
SCENARIO = dict(LOWER_TOL=0.75, MIN_POST=1, K_NEG=5, K_PRE_POS=2)

APOLLO_STATIONS = {
    "S16": {"lat": -8.9730, "lon": 15.5000},
    "S14": {"lat": -3.6440, "lon": -17.4775},
    "S15": {"lat": 26.1322, "lon": 3.6339},
}

# Set to None to keep all selected events.
MAX_EVENTS = None

# 3D spherical grid used by pykonal.
NR = 45
NTHETA = 61
NPHI = 121

# Dipole near/far-side perturbation amplitude.
DEG2_EPS = 0.03
DEG2_VECTOR = np.array([1.0, 0.0, 0.0])

# If True, also solve the same event in the unperturbed 1D model.
COMPUTE_REFERENCE_MODEL = True

# Ray tracing mode: use the native pykonal tracer first.
TRACE_METHOD = "native"
STRICT_NATIVE_TRACE = False

# Steepest-descent tracing step through the pykonal travel-time field.
TRACE_STEP_KM = 2.0

# Debug controls for tracing/plotting a single event.
DEBUG_EVENT_INDEX = 0
DEBUG_EVENT_STARTTIME = None
DEBUG_PRINT_SAMPLES = 12


In [ ]:
def _first_existing_col(cols, candidates):
    for col in candidates:
        if col in cols:
            return col
    return None


def read_vpremoon_model(model_file):
    rows = []
    started = False

    with open(model_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split()

            if not started:
                if len(parts) >= 4 and parts[0].lower() == "depth_km":
                    started = True
                continue

            if len(parts) < 4:
                continue

            try:
                rows.append([
                    float(parts[0]),
                    float(parts[1]),
                    float(parts[2]),
                    float(parts[3]),
                ])
            except ValueError:
                continue

    if not rows:
        raise ValueError(f"No model rows found in file: {model_file}")

    return pd.DataFrame(
        rows,
        columns=["depth_km", "vp_km_s", "vs_km_s", "rho_g_cm3"],
    )


def load_excel_long(xlsx, sheet, *, FC, BANDS):
    d = pd.read_excel(xlsx, sheet_name=sheet)

    need = ["starttime", "station", "fc_hz", "t0_dt_mean"]
    missing = [c for c in need if c not in d.columns]
    if missing:
        raise KeyError(
            f"Missing columns in sheet '{sheet}': {missing}; available: {list(d.columns)}"
        )

    d["station"] = d["station"].astype(str)
    d["fc_hz"] = pd.to_numeric(d["fc_hz"], errors="coerce").astype(float)
    d["starttime_dt"] = pd.to_datetime(d["starttime"], errors="coerce", utc=True)
    d["t0_dt_mean_dt"] = pd.to_datetime(d["t0_dt_mean"], errors="coerce", utc=True)

    if "distance" in d.columns:
        d["distance_deg"] = pd.to_numeric(d["distance"], errors="coerce")
    elif "epi_deg" in d.columns:
        d["distance_deg"] = pd.to_numeric(d["epi_deg"], errors="coerce")
    else:
        d["distance_deg"] = np.nan

    lat_col = _first_existing_col(d.columns, ["lat", "Lat", "LAT", "latitude", "Latitude"])
    lon_col = _first_existing_col(d.columns, ["Lon", "lon", "LON", "longitude", "Longitude"])

    if lat_col is not None:
        d["lat"] = pd.to_numeric(d[lat_col], errors="coerce")
    else:
        d["lat"] = np.nan

    if lon_col is not None:
        d["Lon"] = pd.to_numeric(d[lon_col], errors="coerce")
    else:
        d["Lon"] = np.nan

    d = d[d["fc_hz"].isin(BANDS)].copy()
    d["event"] = d["starttime_dt"].astype(str) + "__" + d["station"]

    ref = (
        d[d["fc_hz"].eq(FC)][["event", "t0_dt_mean_dt"]]
        .rename(columns={"t0_dt_mean_dt": "t0_fc_dt"})
        .groupby("event", as_index=False)["t0_fc_dt"]
        .min()
    )
    d = d.merge(ref, on="event", how="left")
    d["dt_rel"] = (d["t0_dt_mean_dt"] - d["t0_fc_dt"]).dt.total_seconds()
    d = d[d["dt_rel"].notna() & d["starttime_dt"].notna() & d["t0_dt_mean_dt"].notna()].copy()

    return d[[
        "event",
        "station",
        "starttime_dt",
        "fc_hz",
        "dt_rel",
        "distance_deg",
        "lat",
        "Lon",
        "t0_dt_mean_dt",
    ]].rename(columns={"fc_hz": "band"})


def build_event_band_matrix(df_long, *, BANDS):
    return (
        df_long.pivot_table(index="event", columns="band", values="dt_rel", aggfunc="first")
        .reindex(columns=BANDS)
        .sort_index()
    )


def select_events(*, dt_mat, FC, BANDS, MIN_POST, K_NEG, K_PRE_POS=0, LOWER_TOL=0.0):
    post_bands = [b for b in BANDS if b > FC]
    pre_bands = [b for b in BANDS if b < FC]

    keep = []
    for ev in dt_mat.index:
        dt = dt_mat.loc[ev]

        post_vals = dt[post_bands].dropna()
        if len(post_vals) < MIN_POST:
            keep.append(False)
            continue

        if int((post_vals <= LOWER_TOL).sum()) > K_NEG:
            keep.append(False)
            continue

        pre_vals = dt[pre_bands].dropna()
        if int((pre_vals > LOWER_TOL).sum()) > K_PRE_POS:
            keep.append(False)
            continue

        keep.append(True)

    return pd.Series(keep, index=dt_mat.index, name="keep")


def build_selected_events_from_excel():
    df_long = load_excel_long(XLSX, SHEET, FC=FC, BANDS=BANDS)
    dt_mat = build_event_band_matrix(df_long, BANDS=BANDS)
    keep_mask = select_events(dt_mat=dt_mat, FC=FC, BANDS=BANDS, **SCENARIO)

    dist_map = df_long[["event", "distance_deg"]].drop_duplicates(subset=["event"]).set_index("event")["distance_deg"].to_dict()
    lat_map = df_long[["event", "lat"]].drop_duplicates(subset=["event"]).set_index("event")["lat"].to_dict()
    lon_map = df_long[["event", "Lon"]].drop_duplicates(subset=["event"]).set_index("event")["Lon"].to_dict()
    time_map = (
        df_long[df_long["band"] == FC][["event", "t0_dt_mean_dt"]]
        .dropna()
        .drop_duplicates(subset=["event"])
        .set_index("event")["t0_dt_mean_dt"]
        .to_dict()
    )

    rows = []
    for ev in keep_mask.index[keep_mask]:
        rows.append(
            dict(
                event=ev,
                time_utc=time_map.get(ev, pd.NaT),
                station=ev.split("__", 1)[-1],
                epi_deg=dist_map.get(ev, np.nan),
                lat=lat_map.get(ev, np.nan),
                Lon=lon_map.get(ev, np.nan),
            )
        )

    return pd.DataFrame(rows)


def load_event_catalog():
    selected_df = build_selected_events_from_excel()

    if SELECTED_EVENTS_CSV.exists():
        coords_df = pd.read_csv(SELECTED_EVENTS_CSV)
        selected_df = selected_df.drop(columns=[c for c in ["lat", "Lon"] if c in selected_df.columns])
        df = selected_df.merge(
            coords_df[["event", "lat", "Lon"]].drop_duplicates(subset=["event"]),
            on="event",
            how="left",
        )
    else:
        df = selected_df.copy()

    df["station"] = df["station"].astype(str).str.strip()
    for col in ["epi_deg", "lat", "Lon"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    n_before_coords = len(df)
    df = df[
        np.isfinite(df["epi_deg"]) &
        np.isfinite(df["lat"]) &
        np.isfinite(df["Lon"]) &
        df["station"].isin(APOLLO_STATIONS.keys())
    ].copy()

    df = df.sort_values(["epi_deg", "time_utc"], na_position="last").drop_duplicates(subset=["event"])
    print(f"Scenario-selected events before coordinate filtering: {n_before_coords}")
    print(f"Scenario-selected events with usable coordinates: {len(df)}")
    if MAX_EVENTS is not None:
        df = df.head(MAX_EVENTS)
        print(f"Events kept after MAX_EVENTS filter: {len(df)}")

    return df.reset_index(drop=True)


def latlon_to_spherical(lat_deg, lon_deg, depth_km=0.0, radius_km=RMOON_KM):
    return np.array([
        radius_km - depth_km,
        np.deg2rad(90.0 - lat_deg),
        np.deg2rad(lon_deg % 360.0),
    ], dtype=float)


def spherical_to_cartesian(points):
    pts = np.asarray(points, dtype=float)
    r = pts[..., 0]
    theta = pts[..., 1]
    phi = pts[..., 2]

    sin_theta = np.sin(theta)
    x = r * sin_theta * np.cos(phi)
    y = r * sin_theta * np.sin(phi)
    z = r * np.cos(theta)

    return np.stack([x, y, z], axis=-1)


def nearest_node_index(coords, min_coords, node_intervals, npts):
    idx = np.rint((coords - min_coords) / node_intervals).astype(int)
    idx = np.clip(idx, 0, np.asarray(npts, dtype=int) - 1)
    return tuple(idx.tolist())


def degree2_field(theta, phi, vector=DEG2_VECTOR):
    vector = np.asarray(vector, dtype=float)
    vector = vector / np.linalg.norm(vector)
    ux = np.sin(theta) * np.cos(phi)
    uy = np.sin(theta) * np.sin(phi)
    uz = np.cos(theta)
    dot = vector[0] * ux + vector[1] * uy + vector[2] * uz
    return dot


def build_velocity_grid(model_df, deg2_eps=DEG2_EPS):
    depth = model_df["depth_km"].to_numpy(dtype=float)
    vp = model_df["vp_km_s"].to_numpy(dtype=float)

    model_radius = RMOON_KM - depth
    order = np.argsort(model_radius)
    model_radius = model_radius[order]
    vp = vp[order]

    radius_min_km = max(1.0, RMOON_KM - depth.max())
    radii = np.linspace(radius_min_km, RMOON_KM, NR)
    vp_r = np.interp(radii, model_radius, vp)

    theta = np.linspace(0.0, np.pi, NTHETA)
    phi = np.linspace(0.0, 2.0 * np.pi, NPHI)
    Theta, Phi = np.meshgrid(theta, phi, indexing="ij")

    angular_scale = 1.0 + deg2_eps * degree2_field(Theta, Phi)
    velocity = vp_r[:, None, None] * angular_scale[None, :, :]

    return velocity.copy(), radius_min_km


def trilinear_interpolate(field, coords, min_coords, node_intervals):
    coords = np.asarray(coords, dtype=float)
    idx_float = (coords - min_coords) / node_intervals
    idx0 = np.floor(idx_float).astype(int)
    frac = np.clip(idx_float - idx0, 0.0, 1.0)
    max_base = np.asarray(field.shape, dtype=int) - 2
    idx0 = np.clip(idx0, 0, max_base)

    out = 0.0
    for dr in (0, 1):
        wr = (1.0 - frac[0]) if dr == 0 else frac[0]
        for dt in (0, 1):
            wt = (1.0 - frac[1]) if dt == 0 else frac[1]
            for dp in (0, 1):
                wp = (1.0 - frac[2]) if dp == 0 else frac[2]
                out += wr * wt * wp * field[idx0[0] + dr, idx0[1] + dt, idx0[2] + dp]
    return float(out)


def trace_ray_with_pykonal(solver, end_coords, *, min_points=2, min_arc_length_km=1.0):
    ray = np.asarray(solver.trace_ray(np.asarray(end_coords, dtype=float)), dtype=float)

    if ray.ndim != 2 or ray.shape[1] != 3:
        raise ValueError(f"Native pykonal trace_ray returned invalid shape: {ray.shape}")

    if len(ray) < min_points:
        raise ValueError(f"Native pykonal trace_ray returned too few samples: {len(ray)}")

    if not np.all(np.isfinite(ray)):
        raise ValueError("Native pykonal trace_ray returned non-finite values")

    segment_lengths = np.linalg.norm(np.diff(ray, axis=0), axis=1)
    if segment_lengths.sum() < min_arc_length_km:
        raise ValueError(
            "Native pykonal trace_ray returned a degenerate path "
            f"(total length={segment_lengths.sum():.6f} km)"
        )

    return ray


def trace_ray_from_gradient(field, end_coords, src_coords, step_km=TRACE_STEP_KM, max_steps=4000):
    coords = np.asarray(end_coords, dtype=float).copy()
    src = np.asarray(src_coords, dtype=float)
    path = [coords.copy()]
    value = np.inf

    for _ in range(max_steps):
        previous_value = value
        value = field.value(coords)
        if value >= previous_value or np.isnan(value):
            path.pop()
            break

        if np.linalg.norm(coords - src) <= 2.0 * step_km:
            path.append(src.copy())
            break

        grad = np.asarray(field.gradient.value(coords), dtype=float)
        grad_norm = np.linalg.norm(grad)
        if not np.isfinite(grad_norm) or grad_norm < 1e-12:
            break

        grad /= grad_norm
        grad[1] /= coords[0]
        grad[2] /= max(coords[0] * np.sin(coords[1]), 1e-6)

        new_coords = coords - step_km * grad
        new_coords[2] = new_coords[2] % (2.0 * np.pi)

        if np.linalg.norm(new_coords - coords) < 1e-8:
            break

        coords = new_coords
        path.append(coords.copy())

    return np.vstack(path)


def solve_event_with_pykonal(row, model_df, deg2_eps=DEG2_EPS):
    velocity, radius_min_km = build_velocity_grid(model_df, deg2_eps=deg2_eps)

    dr = (RMOON_KM - radius_min_km) / (NR - 1)
    dtheta = np.pi / (NTHETA - 1)
    dphi = 2.0 * np.pi / (NPHI - 1)

    solver = pykonal.EikonalSolver(coord_sys="spherical")
    solver.velocity.min_coords = (radius_min_km, 0.0, 0.0)
    solver.velocity.node_intervals = (dr, dtheta, dphi)
    solver.velocity.npts = (NR, NTHETA, NPHI)
    solver.velocity.values = velocity

    src_coords = latlon_to_spherical(float(row["lat"]), float(row["Lon"]), SOURCE_DEPTH_KM)
    station = APOLLO_STATIONS[row["station"]]
    rec_coords = latlon_to_spherical(station["lat"], station["lon"], 1.0)

    min_coords = np.asarray(solver.velocity.min_coords, dtype=float)
    node_intervals = np.asarray(solver.velocity.node_intervals, dtype=float)

    src_idx = nearest_node_index(src_coords, min_coords, node_intervals, solver.velocity.npts)
    solver.traveltime.values[src_idx] = 0.0
    solver.unknown[src_idx] = False
    solver.trial.push(*src_idx)
    solver.solve()

    rec_idx = nearest_node_index(rec_coords, min_coords, node_intervals, solver.velocity.npts)
    travel_time_s = float(solver.traveltime.values[rec_idx])

    trace_method = TRACE_METHOD
    if TRACE_METHOD == "native":
        try:
            ray_rtp = trace_ray_with_pykonal(solver, rec_coords)
        except Exception as exc:
            if STRICT_NATIVE_TRACE:
                raise
            print(f"Native pykonal trace_ray failed for {row['event']}: {exc}")
            ray_rtp = trace_ray_from_gradient(solver.traveltime, rec_coords, src_coords)
            trace_method = "gradient_fallback"
    elif TRACE_METHOD == "gradient":
        ray_rtp = trace_ray_from_gradient(solver.traveltime, rec_coords, src_coords)
    else:
        raise ValueError(f"Unsupported TRACE_METHOD: {TRACE_METHOD}")

    return dict(
        event=row["event"],
        station=row["station"],
        epi_deg=float(row["epi_deg"]),
        src_lat=float(row["lat"]),
        src_lon=float(row["Lon"]),
        travel_time_s=travel_time_s,
        deg2_eps=deg2_eps,
        ray_rtp=ray_rtp,
        ray_xyz=spherical_to_cartesian(ray_rtp),
        receiver_rtp=rec_coords,
        trace_method=trace_method,
    )


In [ ]:
def plot_degree2_field_with_rays(catalog_df, solutions, deg2_eps=DEG2_EPS, nlat=181, nlon=361):
    lat_deg = np.linspace(-90.0, 90.0, nlat)
    lon_deg = np.linspace(-180.0, 180.0, nlon)
    Lon_deg, Lat_deg = np.meshgrid(lon_deg, lat_deg)

    theta = np.deg2rad(90.0 - Lat_deg)
    phi = np.deg2rad(Lon_deg % 360.0)
    field = deg2_eps * degree2_field(theta, phi)

    fig = plt.figure(figsize=(16, 6), dpi=140)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.0])
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1], projection="3d")

    pcm = ax1.pcolormesh(Lon_deg, Lat_deg, field, cmap="coolwarm", shading="auto")
    ax1.contour(Lon_deg, Lat_deg, field, levels=12, colors="k", linewidths=0.45, alpha=0.35)
    cbar = fig.colorbar(pcm, ax=ax1, fraction=0.046, pad=0.02)
    cbar.set_label("Dipole perturbation scale")

    for sta, info in APOLLO_STATIONS.items():
        ax1.scatter(info["lon"], info["lat"], marker="^", s=80, color="limegreen", edgecolor="k", zorder=4)
        ax1.text(info["lon"], info["lat"], f" {sta}", fontsize=9, weight="bold")

    for sol in solutions:
        src_lon = sol["src_lon"]
        src_lat = sol["src_lat"]
        rec = APOLLO_STATIONS[sol["station"]]
        ax1.scatter(src_lon, src_lat, s=28, color="gold", edgecolor="k", zorder=5)

        ray = sol["ray_rtp"]
        ray_lat = 90.0 - np.rad2deg(ray[:, 1])
        ray_lon = ((np.rad2deg(ray[:, 2]) + 180.0) % 360.0) - 180.0
        ax1.plot(ray_lon, ray_lat, color="0.3", lw=1.0, alpha=0.7)
        ax1.plot([src_lon, rec["lon"]], [src_lat, rec["lat"]], color="0.3", lw=0.6, alpha=0.25)

    ax1.set_xlim(-180, 180)
    ax1.set_ylim(-90, 90)
    ax1.set_xlabel("Longitude (deg)")
    ax1.set_ylabel("Latitude (deg)")
    ax1.set_title("Dipole perturbation field + rays/stations")
    ax1.grid(alpha=0.25)

    uu = np.linspace(0.0, 2.0 * np.pi, 180)
    vv = np.linspace(-0.5 * np.pi, 0.5 * np.pi, 90)
    UU, VV = np.meshgrid(uu, vv)
    XX = np.cos(VV) * np.cos(UU)
    YY = np.cos(VV) * np.sin(UU)
    ZZ = np.sin(VV)
    surf_field = deg2_eps * degree2_field(0.5 * np.pi - VV, UU)
    norm = plt.Normalize(vmin=surf_field.min(), vmax=surf_field.max())
    ax2.plot_surface(XX, YY, ZZ, facecolors=plt.cm.coolwarm(norm(surf_field)), linewidth=0, antialiased=True, shade=False, alpha=0.92)

    for sta, info in APOLLO_STATIONS.items():
        st = latlon_to_spherical(info["lat"], info["lon"], 0.0, radius_km=1.0)
        st_xyz = spherical_to_cartesian(st)
        ax2.scatter(st_xyz[0], st_xyz[1], st_xyz[2], marker="^", color="k", s=45)
        ax2.text(st_xyz[0], st_xyz[1], st_xyz[2], f" {sta}", fontsize=9)

    for sol in solutions:
        src = latlon_to_spherical(sol["src_lat"], sol["src_lon"], 0.0, radius_km=1.0)
        src_xyz = spherical_to_cartesian(src)
        ax2.scatter(src_xyz[0], src_xyz[1], src_xyz[2], color="gold", edgecolor="k", s=28)

        ray_unit = sol["ray_xyz"] / np.maximum(np.linalg.norm(sol["ray_xyz"], axis=1, keepdims=True), 1e-12)
        ax2.plot(ray_unit[:, 0], ray_unit[:, 1], ray_unit[:, 2], color="0.15", lw=1.1)

    ax2.set_box_aspect((1, 1, 1))
    ax2.set_xlabel("X")
    ax2.set_ylabel("Y")
    ax2.set_zlabel("Z")
    ax2.set_title("Field painted on unit sphere + rays/stations")
    plt.tight_layout()
    plt.show()


In [ ]:
def summarize_ray_debug(solution, print_samples=DEBUG_PRINT_SAMPLES):
    ray_rtp = np.asarray(solution["ray_rtp"], dtype=float)
    ray_xyz = np.asarray(solution["ray_xyz"], dtype=float)

    ray_lat_deg = 90.0 - np.rad2deg(ray_rtp[:, 1])
    ray_lon_deg = ((np.rad2deg(ray_rtp[:, 2]) + 180.0) % 360.0) - 180.0
    ray_depth_km = RMOON_KM - ray_rtp[:, 0]

    diffs_xyz = np.diff(ray_xyz, axis=0)
    step_xyz_km = np.linalg.norm(diffs_xyz, axis=1) if len(ray_xyz) > 1 else np.array([], dtype=float)
    diffs_rtp = np.diff(ray_rtp, axis=0)

    print(f"Event:   {solution['event']}")
    print(f"Station: {solution['station']}")
    print(f"Travel time (perturbed): {solution['travel_time_s']:.3f} s")
    print(f"Ray samples: {len(ray_rtp)}")
    print(f"Depth range along traced ray: {ray_depth_km.min():.2f} km to {ray_depth_km.max():.2f} km")
    if step_xyz_km.size:
        print(
            "Step length in XYZ (km): "
            f"min={step_xyz_km.min():.3f}, median={np.median(step_xyz_km):.3f}, max={step_xyz_km.max():.3f}"
        )
    if len(ray_rtp) > 1:
        print(
            "RTP increments: "
            f"dr range=({diffs_rtp[:,0].min():.3f}, {diffs_rtp[:,0].max():.3f}), "
            f"dtheta range=({diffs_rtp[:,1].min():.6f}, {diffs_rtp[:,1].max():.6f}), "
            f"dphi range=({diffs_rtp[:,2].min():.6f}, {diffs_rtp[:,2].max():.6f})"
        )

    debug_df = pd.DataFrame({
        "r_km": ray_rtp[:, 0],
        "theta_rad": ray_rtp[:, 1],
        "phi_rad": ray_rtp[:, 2],
        "lat_deg": ray_lat_deg,
        "lon_deg": ray_lon_deg,
        "depth_km": ray_depth_km,
        "x_km": ray_xyz[:, 0],
        "y_km": ray_xyz[:, 1],
        "z_km": ray_xyz[:, 2],
    })

    n_head = min(print_samples, len(debug_df))
    n_tail = min(print_samples, len(debug_df))
    print("\nFirst samples:")
    display(debug_df.head(n_head))
    print("\nLast samples:")
    display(debug_df.tail(n_tail))

    return debug_df


def plot_single_ray_debug(solution, show_surface_projection=True):
    ray_rtp = np.asarray(solution["ray_rtp"], dtype=float)
    ray_xyz = np.asarray(solution["ray_xyz"], dtype=float)
    ray_lat_deg = 90.0 - np.rad2deg(ray_rtp[:, 1])
    ray_lon_deg = ((np.rad2deg(ray_rtp[:, 2]) + 180.0) % 360.0) - 180.0
    ray_depth_km = RMOON_KM - ray_rtp[:, 0]

    src_xyz = ray_xyz[-1]
    rec_xyz = ray_xyz[0]

    src_unit = src_xyz / np.linalg.norm(src_xyz)
    rec_unit = rec_xyz / np.linalg.norm(rec_xyz)
    plane_e1 = src_unit
    plane_e2 = rec_unit - np.dot(rec_unit, plane_e1) * plane_e1
    plane_e2_norm = np.linalg.norm(plane_e2)
    if plane_e2_norm < 1e-10:
        plane_e2 = np.array([0.0, 0.0, 1.0])
    else:
        plane_e2 = plane_e2 / plane_e2_norm

    ray_plane_x = ray_xyz @ plane_e1
    ray_plane_y = ray_xyz @ plane_e2
    src_plane = np.array([src_xyz @ plane_e1, src_xyz @ plane_e2])
    rec_plane = np.array([rec_xyz @ plane_e1, rec_xyz @ plane_e2])

    fig = plt.figure(figsize=(18, 10), dpi=140)
    gs = fig.add_gridspec(2, 3, height_ratios=[1.1, 0.9])
    ax3d = fig.add_subplot(gs[0, 0], projection="3d")
    ax_plane = fig.add_subplot(gs[0, 1])
    ax_xz = fig.add_subplot(gs[0, 2])
    ax_yz = fig.add_subplot(gs[1, 0])
    ax_map = fig.add_subplot(gs[1, 1:])

    t = np.linspace(0.0, 2.0 * np.pi, 400)
    u = np.linspace(0.0, 2.0 * np.pi, 120)
    v = np.linspace(-0.5 * np.pi, 0.5 * np.pi, 60)
    UU, VV = np.meshgrid(u, v)
    XS = RMOON_KM * np.cos(VV) * np.cos(UU)
    YS = RMOON_KM * np.cos(VV) * np.sin(UU)
    ZS = RMOON_KM * np.sin(VV)

    ax3d.plot_surface(XS, YS, ZS, alpha=0.08, color="0.7", linewidth=0, shade=False)
    ax3d.plot(ray_xyz[:, 0], ray_xyz[:, 1], ray_xyz[:, 2], color="tab:blue", lw=2.0)
    ax3d.scatter(src_xyz[0], src_xyz[1], src_xyz[2], color="gold", edgecolor="k", s=45, label="Source")
    ax3d.scatter(rec_xyz[0], rec_xyz[1], rec_xyz[2], color="tab:red", edgecolor="k", s=45, label="Receiver")
    ax3d.set_title("Single traced ray in 3D")
    ax3d.set_box_aspect((1, 1, 1))
    ax3d.set_xlabel("X (km)")
    ax3d.set_ylabel("Y (km)")
    ax3d.set_zlabel("Z (km)")
    ax3d.legend(loc="upper left")

    ax_plane.plot(RMOON_KM * np.cos(t), RMOON_KM * np.sin(t), color="0.7")
    ax_plane.plot(ray_plane_x, ray_plane_y, color="tab:blue", lw=2.0)
    ax_plane.scatter(src_plane[0], src_plane[1], color="gold", edgecolor="k", s=35)
    ax_plane.scatter(rec_plane[0], rec_plane[1], color="tab:red", edgecolor="k", s=35)
    ax_plane.set_aspect("equal")
    ax_plane.set_title("Ray in source-receiver plane")
    ax_plane.set_xlabel("Plane axis 1 (km)")
    ax_plane.set_ylabel("Plane axis 2 (km)")

    ax_xz.plot(RMOON_KM * np.cos(t), RMOON_KM * np.sin(t), color="0.7")
    ax_xz.plot(ray_xyz[:, 0], ray_xyz[:, 2], color="tab:blue", lw=2.0)
    ax_xz.scatter(src_xyz[0], src_xyz[2], color="gold", edgecolor="k", s=35)
    ax_xz.scatter(rec_xyz[0], rec_xyz[2], color="tab:red", edgecolor="k", s=35)
    ax_xz.set_aspect("equal")
    ax_xz.set_title("Same ray projected in X-Z")
    ax_xz.set_xlabel("X (km)")
    ax_xz.set_ylabel("Z (km)")

    ax_yz.plot(RMOON_KM * np.cos(t), RMOON_KM * np.sin(t), color="0.7")
    ax_yz.plot(ray_xyz[:, 1], ray_xyz[:, 2], color="tab:blue", lw=2.0)
    ax_yz.scatter(src_xyz[1], src_xyz[2], color="gold", edgecolor="k", s=35)
    ax_yz.scatter(rec_xyz[1], rec_xyz[2], color="tab:red", edgecolor="k", s=35)
    ax_yz.set_aspect("equal")
    ax_yz.set_title("Same ray projected in Y-Z")
    ax_yz.set_xlabel("Y (km)")
    ax_yz.set_ylabel("Z (km)")

    if show_surface_projection:
        ax_map.plot(ray_lon_deg, ray_lat_deg, color="tab:blue", lw=1.7, label="Projected traced ray")
        ax_map.scatter(solution["src_lon"], solution["src_lat"], color="gold", edgecolor="k", s=35, label="Source")
        rec = APOLLO_STATIONS[solution["station"]]
        ax_map.scatter(rec["lon"], rec["lat"], color="tab:red", edgecolor="k", s=35, label="Receiver")
        ax_map.set_title("Angular coordinates of interior ray samples")
        ax_map.set_xlabel("Longitude (deg)")
        ax_map.set_ylabel("Latitude (deg)")
        ax_map.grid(alpha=0.25)
        ax_map.legend(loc="best", fontsize=8)

    fig.suptitle(f"Debug ray: {solution['event']} -> {solution['station']}", y=0.98)
    plt.tight_layout()
    plt.show()

    fig2, ax = plt.subplots(1, 1, figsize=(7, 4), dpi=140)
    ax.plot(np.arange(len(ray_depth_km)), ray_depth_km, color="tab:purple")
    ax.set_xlabel("Path sample index")
    ax.set_ylabel("Depth (km)")
    ax.invert_yaxis()
    ax.set_title("Depth evolution along traced ray")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
def pick_debug_event_index(catalog_df, debug_event_index=DEBUG_EVENT_INDEX, debug_event_starttime=DEBUG_EVENT_STARTTIME):
    if debug_event_starttime is None:
        return int(debug_event_index)

    starttime_str = str(debug_event_starttime).strip()
    event_start = catalog_df["event"].astype(str).str.split("__", n=1).str[0]
    matches = np.where(event_start == starttime_str)[0]

    if len(matches) == 0:
        raise KeyError(
            f"No event found for DEBUG_EVENT_STARTTIME={starttime_str!r}. "
            f"Available starttimes include: {event_start.head(10).tolist()}"
        )

    return int(matches[0])


In [ ]:
catalog_df = load_event_catalog()
model_df = read_vpremoon_model(MODEL_FILE)

print(f"Events loaded: {len(catalog_df)}")
display(catalog_df[["event", "station", "epi_deg", "lat", "Lon"]])

solutions = []
reference_solutions = []
for row in catalog_df.to_dict(orient="records"):
    solutions.append(solve_event_with_pykonal(row, model_df, deg2_eps=DEG2_EPS))
    if COMPUTE_REFERENCE_MODEL:
        reference_solutions.append(solve_event_with_pykonal(row, model_df, deg2_eps=0.0))

reference_lookup = {sol["event"]: sol for sol in reference_solutions}

results_df = pd.DataFrame([
    {
        "event": sol["event"],
        "station": sol["station"],
        "epi_deg": sol["epi_deg"],
        "src_lat": sol["src_lat"],
        "src_lon": sol["src_lon"],
        "travel_time_ref_s": reference_lookup[sol["event"]]["travel_time_s"] if COMPUTE_REFERENCE_MODEL else np.nan,
        "travel_time_deg2_s": sol["travel_time_s"],
        "delta_t_s": sol["travel_time_s"] - reference_lookup[sol["event"]]["travel_time_s"] if COMPUTE_REFERENCE_MODEL else np.nan,
        "phase": PHASE_NAME,
    }
    for sol in solutions
]).sort_values(["epi_deg", "station"]).reset_index(drop=True)
results_df["travel_time_s"] = results_df["travel_time_deg2_s"]

display(results_df)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), dpi=140)
colors = plt.cm.viridis(np.linspace(0.0, 1.0, len(solutions)))
theta = np.linspace(0.0, 2.0 * np.pi, 400)

for ax, pair in zip(axes[:2], [(0, 2, "X", "Z"), (1, 2, "Y", "Z")]):
    ax.plot(RMOON_KM * np.cos(theta), RMOON_KM * np.sin(theta), color="0.7", lw=1.2)

    for color, sol in zip(colors, solutions):
        ray_xyz = sol["ray_xyz"]
        ax.plot(ray_xyz[:, pair[0]], ray_xyz[:, pair[1]], color=color, lw=1.6)
        ax.scatter(ray_xyz[-1, pair[0]], ray_xyz[-1, pair[1]], color=color, s=20)

    for station_name, info in APOLLO_STATIONS.items():
        st_xyz = spherical_to_cartesian(latlon_to_spherical(info["lat"], info["lon"], 0.0))
        ax.scatter(st_xyz[pair[0]], st_xyz[pair[1]], marker="^", color="k", s=35)
        ax.text(st_xyz[pair[0]], st_xyz[pair[1]], f" {station_name}", fontsize=8)

    ax.set_aspect("equal")
    ax.set_xlabel(f"{pair[2]} (km)")
    ax.set_ylabel(f"{pair[3]} (km)")

axes[0].set_title("Raypaths in X-Z")
axes[1].set_title("Raypaths in Y-Z")

axes[2].scatter(
    results_df["epi_deg"],
    results_df["travel_time_deg2_s"],
    c=np.arange(len(results_df)),
    cmap="viridis",
    s=45,
    label="With dipole perturbation",
)
if COMPUTE_REFERENCE_MODEL:
    axes[2].scatter(
        results_df["epi_deg"],
        results_df["travel_time_ref_s"],
        facecolors="none",
        edgecolors="k",
        s=42,
        label="Reference 1D",
    )
for row in results_df.itertuples(index=False):
    axes[2].text(row.epi_deg + 0.05, row.travel_time_deg2_s, row.station, fontsize=8)

axes[2].set_xlabel("Epicentral distance (deg)")
axes[2].set_ylabel("Travel time (s)")
axes[2].set_title("Pykonal travel times")
axes[2].grid(alpha=0.3)
axes[2].legend(loc="best", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
debug_event_idx = pick_debug_event_index(catalog_df)
print(f"Debug event index used: {debug_event_idx}")
debug_solution = solutions[debug_event_idx]
debug_df = summarize_ray_debug(debug_solution)
plot_single_ray_debug(debug_solution)


In [ ]:
plot_degree2_field_with_rays(catalog_df, solutions, deg2_eps=DEG2_EPS)
